In [ ]:
from accessx.aoi import load_aoi, make_hex_grid
from accessx.graph import build_network, save_graph, load_graph
from accessx.cost import add_time_cost_constant_speed
from accessx.cost import add_slope_based_time, add_edge_cost
from accessx.isochrone import calculate_isochrones

### AOI & Hexes

In [2]:
bbox_milano = (
    9.1860,   # minx (lon)
    45.4620,  # miny (lat)
    9.1960,   # maxx (lon)
    45.4680,  # maxy (lat)
)
city_epsg = 32632
buffer=100


aoi = load_aoi(bbox=bbox_milano, buffer_m=buffer, utm_crs=city_epsg, save_path="data/area/area_of_interest.geojson")
hexes = make_hex_grid(aoi,resolution=10, clip=True)
hexes_proj = hexes.to_crs(city_epsg)

### Get street network

In [3]:

city_epsg = 32632
buffer=100

G_proj = build_network(
    AOI=aoi,
    city_epsg=city_epsg,
    buffer_m=buffer,
    network_type="walk",
    simplify=False,
    retain_all=True,
)


save_graph(
    G_proj,
    out_dir= "data/street_network/",
    base_name="original_graph",
    save_nodes=True,
    save_edges=True,
)

### Calculate cost per street segment 

The cost can be the time to traverse a street based on a defined constant speed...

In [4]:

# time-cost based on costant speed defined by the user
G_proj = add_time_cost_constant_speed(G_proj, speed_kmh=4.5, cost_col="avg_time")


...or a the time when using speed adjusted to the slope of the streets (in %, assuming slope_pct has been added to the graph)...

In [5]:

# custom cost defined by specific function that receives edge as input and returns cost
# example with function that returns time-cost based on slope-related speeds

G_proj = add_edge_cost(
    G_proj,
    cost_fn=add_slope_based_time(slope_col="slope_pct"),
    cost_col="slope_based_time",
)


... or your own cost function..

In [6]:
def discomfort(edge):
    bad = 1.0 if edge.get("lit") == "yes" else 1.5
    return edge["length"] * bad

G_proj = add_edge_cost(G_proj, cost_fn=discomfort, cost_col="discomfort")

In [7]:
save_graph(
    G_proj,
    out_dir= "data/street_network/",
    base_name="graph_with_cost",
    save_nodes=False,
    save_edges=True,
)

city_epsg = 32632

G_proj = load_graph(out_dir="data/street_network/", base_name="graph_with_cost", crs=city_epsg)

# G_proj = load_graph(
#     nodes_path="data/street_network/graph_with_cost_nodes_OSM.geojson",
#     edges_path="data/street_network/graph_with_cost_edges_OSM.geojson",
#     crs=3044
# )

save_graph(
    G_proj,
    out_dir= "data/street_network/",
    base_name="loaded_saved_graph",
    save_nodes=False,
    save_edges=True,
)

## Calculate isochrones

In [8]:
walksheds = make_walksheds(
    G_proj,
    hexes_proj,  # hex polygons are ok; function uses centroids
    cost_thresholds=[5, 10, 15],
    cost_attr="time_min",
    city_epsg=3044,
    max_distance=200,
    method="edges",   # or "hull"
    edge_buff=25,
    infill=True,
    save_dir="data/walksheds/"
)

/Users/vmlias/repos/accessX/src/accessx/isochrone.py:143: UserWarning: Geometry column does not contain geometry.
  out_csv[col] = out_csv[col].apply(
/Users/vmlias/repos/accessX/src/accessx/isochrone.py:143: UserWarning: Geometry column does not contain geometry.
  out_csv[col] = out_csv[col].apply(
/Users/vmlias/repos/accessX/src/accessx/isochrone.py:143: UserWarning: Geometry column does not contain geometry.
  out_csv[col] = out_csv[col].apply(


In [9]:
walksheds = make_walksheds(
    G_proj,
    hexes_proj,  # hex polygons are ok; function uses centroids
    cost_thresholds=[5, 10, 15],
    cost_attr="time_min",
    city_epsg=3044,
    max_distance=200,
    method="hull",   # or "hull"
    edge_buff=25,
    infill=True,
    save_dir="data/walksheds"
)

/Users/vmlias/repos/accessX/src/accessx/isochrone.py:143: UserWarning: Geometry column does not contain geometry.
  out_csv[col] = out_csv[col].apply(
/Users/vmlias/repos/accessX/src/accessx/isochrone.py:143: UserWarning: Geometry column does not contain geometry.
  out_csv[col] = out_csv[col].apply(
/Users/vmlias/repos/accessX/src/accessx/isochrone.py:143: UserWarning: Geometry column does not contain geometry.
  out_csv[col] = out_csv[col].apply(


In [ ]:
from accessx.poi import get_pois_osm

# example
tags_library = {
    "greenspace": {
        "leisure": ["park", "nature_reserve"],
        "landuse": ["recreation_ground", "grass", "forest"],
        "natural": ["wood", "grassland", "scrub", "heath", "hill"],
    },
    "playground": {"leisure": ["playground"]},
    "supermarket": {"shop": ["supermarket"]},
    "bakery": {"shop": ["bakery"]},
    "butcher": {"shop": ["butcher"]},
    "greengrocer": {"shop": ["greengrocer"]},
    "seafood": {"shop": ["seafood"]},
    "school": {"amenity": ["kindergarten", "school"]},
    "culture": {"tourism": ["museum", "gallery", "artwork"], "amenity": ["arts_centre"]},
    "sport": {
        "leisure": ["sports_centre", "stadium", "sports_hall", "swimming_pool", "fitness_centre",
                    "fitness_station", "pitch", "track"],
        "sport": True,
    },
    "pharmacy": {"amenity": ["pharmacy"]},
    "nightlife": {
        "amenity": ["bar", "pub", "nightclub", "music_venue", "cinema", "theatre"],
        "tourism": ["nightlife"],
    },
    "metro": {"station": ["subway"]},
    "bus_tram": {
        "highway": ["bus_stop"],
        "railway": ["tram_stop"],
        "amenity": ["bus_station"],
        "public_transport": ["stop_position"],
    },
    "public_square": {"place": ["square"], "amenity": ["marketplace", "town_square"]},
    "cafe_restaurant": {"amenity": ["cafe", "restaurant", "fast_food"]},
    "clothing_stores": {"shop": ["clothes", "boutique", "fashion_accessories", "outfitter", "tailor", "second_hand"]},
    "university": {"amenity": ["college", "university"]},
    "library": {"amenity": ["library"]},
}





In [ ]:
# clean default
pois = get_pois_osm(aoi, categories=["pharmacy", "school"])

,id,osmid,osm_type,category,geometry
0,0,1419998497,node,clothing_stores,POINT (9.18733 45.46283)
1,1,1499956281,node,clothing_stores,POINT (9.18766 45.46339)
2,2,1607530434,node,clothing_stores,POINT (9.18706 45.46469)
3,3,1678582972,node,clothing_stores,POINT (9.19309 45.46488)
4,4,1678582982,node,clothing_stores,POINT (9.19384 45.46507)
...,...,...,...,...,...
189,189,13227393714,node,clothing_stores,POINT (9.18856 45.46626)
190,190,13236560050,node,clothing_stores,POINT (9.19019 45.46642)
191,191,13249380396,node,clothing_stores,POINT (9.19368 45.46533)
192,192,13260948411,node,clothing_stores,POINT (9.18758 45.46401)
